In [ ]:
# Cài đặt thư viện (nếu chưa có)


import os,sys
import gc
import time
import logging
from dataclasses import dataclass

import numpy as np
import polars as pl
import scipy.sparse as sp

from google.colab import drive
drive.mount('/content/drive')

env_path = '/content/drive/My Drive/Thesis/book_recsys/libraries/'
sys.path.append(os.path.abspath('/content/drive/My Drive/Thesis/book_recsys'))

sys.path.insert(0, env_path)
import implicit
print(implicit.__version__)


# Thiết lập log
logging.basicConfig(level=logging.INFO,force=True, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# Bật OpenMP multi-threading để CPU chạy nhanh hơn ở khâu chuẩn bị dữ liệu
os.environ['OPENBLAS_NUM_THREADS'] = '1'





Mounted at /content/drive
0.7.2


In [ ]:
!pip install  polars numpy scipy tqdm

In [ ]:
@dataclass
class ALSConfig:
    # Siêu tham số Mô hình
    factors: int           = 16     # Ép bằng đúng số chiều của SBERT (để sau này dễ kết hợp)
    iterations: int        = 20      # Số vòng lặp giải phương trình (15 là chuẩn công nghiệp)
    regularization: float  = 0.1    # Tránh Overfitting
    alpha: float           = 40.0    # Trọng số phóng đại độ tin cậy (Confidence scaling)

    # Đường dẫn
    base_dir: str          = '/content/drive/My Drive/Thesis/book_recsys'
    version = 'v1'
    data_dir = f"{base_dir}/data/processed_2"
    model_dir =  f"{data_dir}/model_{version}"
    out_dir = f'{data_dir}/model_v3'

    def __post_init__(self):
        data_dir = f"{self.base_dir}/data/processed_2"

        # INPUT: Dữ liệu thô và Index từ hệ thống cũ
        self.path_interactions = f"{data_dir}/train_main.csv"

        #model path
        version = 'v1'
        model_dir = f"{data_dir}/model_{version}"

        self.path_user_mapping = f"{model_dir}/user_index.csv"
        self.path_item_mapping = f"{model_dir}/cb_sbert_book_index.csv"
        if not os.path.exists(self.path_user_mapping):
            print(f'{self.path_user_mapping}')

        # OUTPUT: Vector sinh ra từ ALS
        out_dir = f'{data_dir}/model_v3'
        self.path_out_user_vectors = f"{out_dir}/cf_als_user_profiles.npy"
        self.path_out_item_vectors = f"{out_dir}/cf_als_item_matrix.npy"

cfg = ALSConfig()

In [ ]:
import os
import gc
import duckdb
import implicit
import numpy as np
import scipy.sparse as sp
import pyarrow.parquet as pq

class CFBuilder:
    """
    Enterprise-grade Collaborative Filtering Builder for 100M+ interactions.
    Utilizes DuckDB for out-of-core joins and Memory-Mapped arrays to guarantee Zero-OOM.
    """

    @staticmethod
    def build_als_model(
        interactions_path: str,
        user_index_path: str,
        item_index_path: str,
        output_user_matrix: str,
        output_item_matrix: str,
        temp_dir: str = "output/temp_bigdata/",
        latent_factors: int = 64,
        regularization: float = 0.1,
        iterations: int = 20,
        alpha_val: float = 15.0
    ):
        print(f"INITIATING BIG DATA ALS PIPELINE (100M+ ROWS)\n{'='*50}")
        os.makedirs(temp_dir, exist_ok=True)
        temp_parquet = os.path.join(temp_dir, "mapped_interactions.parquet")

        # ==========================================
        # PHASE 1: DUCKDB OUT-OF-CORE JOIN & MATH
        # ==========================================
        print("\n[PHASE 1] Executing DuckDB Out-of-Core JOIN & Math Calculation...")

        inter_ext = os.path.splitext(interactions_path)[1].lower()
        read_func = "read_parquet" if inter_ext == '.parquet' else "read_csv_auto"

        conn = duckdb.connect(database=':memory:')

        try:
            # --- ĐIỂM SỬA LỖI: Tự động phát hiện tên cột ---
            print(" -> Inspecting schema for item mapping...")
            cols_info = conn.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{item_index_path}') LIMIT 1").fetchall()
            col_names = [info[0] for info in cols_info]

            # Chọn tên cột linh hoạt
            target_item_col = "item_idx" if "item_idx" in col_names else "row_idx"
            print(f" -> Detected item index column: '{target_item_col}'")

            # Câu lệnh SQL khổng lồ đã được tiêm biến target_item_col
            query = f"""
                COPY (
                    SELECT
                        CAST(u.user_idx AS INTEGER) AS u_idx,
                        CAST(i.{target_item_col} AS INTEGER) AS i_idx,
                        CAST(1.0 + ({alpha_val} * COALESCE(r.rating, 1.0)) AS REAL) AS weight
                    FROM {read_func}('{interactions_path}') AS r
                    INNER JOIN read_csv_auto('{user_index_path}') AS u ON r.user_id = u.user_id
                    INNER JOIN read_csv_auto('{item_index_path}') AS i ON r.book_id = i.book_id
                ) TO '{temp_parquet}' (FORMAT 'parquet');
            """

            conn.execute(query)
            print(f" -> DuckDB mapping complete! Saved clean dataset to: {temp_parquet}")

        except Exception as e:
            print(f"DuckDB Error: {e}")
            raise
        finally:
            conn.close()
        # ==========================================
        # PHASE 2: MEMORY MAPPING (np.memmap)
        # ==========================================
        print("\n[PHASE 2] Streaming Parquet to Disk-Backed Memmap Arrays...")
        parquet_file = pq.ParquetFile(temp_parquet)
        total_rows = parquet_file.metadata.num_rows
        print(f" -> Total valid interactions: {total_rows:,} rows")

        # Khai báo file mảng ảo trên ổ cứng
        row_memmap_path = os.path.join(temp_dir, "rows.dat")
        col_memmap_path = os.path.join(temp_dir, "cols.dat")
        data_memmap_path = os.path.join(temp_dir, "weights.dat")

        # Cấp phát dung lượng ổ cứng (Không tốn RAM)
        u_idx_arr = np.memmap(row_memmap_path, dtype=np.int32, mode='w+', shape=(total_rows,))
        i_idx_arr = np.memmap(col_memmap_path, dtype=np.int32, mode='w+', shape=(total_rows,))
        weight_arr = np.memmap(data_memmap_path, dtype=np.float32, mode='w+', shape=(total_rows,))

        # Rót dữ liệu từ Parquet sang Memmap theo từng lô (Batch)
        current_idx = 0
        for batch in parquet_file.iter_batches(batch_size=5_000_000):
            df_batch = batch.to_pandas()
            batch_len = len(df_batch)

            u_idx_arr[current_idx : current_idx + batch_len] = df_batch['u_idx'].values
            i_idx_arr[current_idx : current_idx + batch_len] = df_batch['i_idx'].values
            weight_arr[current_idx : current_idx + batch_len] = df_batch['weight'].values

            current_idx += batch_len
            print(f" -> Memmapped: {current_idx:,} / {total_rows:,} rows", end='\r')

            del df_batch
            gc.collect()

        print("\n -> Memmap flush complete.")

        # Flush bộ nhớ đệm xuống ổ cứng
        u_idx_arr.flush()
        i_idx_arr.flush()
        weight_arr.flush()

        # ==========================================
        # PHASE 3: SPARSE MATRIX & ALS TRAINING
        # ==========================================
        print("\n[PHASE 3] Building CSR Sparse Matrix...")

        # Lấy kích thước tối đa để tạo ma trận
        num_users = u_idx_arr.max() + 1
        num_items = i_idx_arr.max() + 1
        print(f" -> Matrix constraints: {num_users:,} Users x {num_items:,} Items")

        # Cấp phát CSR Matrix (Khoảng ~1.5 GB RAM cho 170M dòng)
        sparse_matrix = sp.csr_matrix(
            (weight_arr, (u_idx_arr, i_idx_arr)),
            shape=(num_users, num_items)
        )
        print(" -> CSR Matrix built successfully in RAM.")

        # Xóa các tham chiếu memmap để hệ điều hành tự do thu hồi bộ nhớ
        del u_idx_arr, i_idx_arr, weight_arr
        gc.collect()

        print(f"\n[PHASE 4] Initializing ALS Engine (Factors: {latent_factors})...")
        # Ép CPU chạy đơn luồng cho toán học lõi để tối ưu Cython của thư viện implicit
        os.environ['OPENBLAS_NUM_THREADS'] = '1'

        model = implicit.als.AlternatingLeastSquares(
            factors=latent_factors,
            regularization=regularization,
            iterations=iterations,
            calculate_training_loss=True,
            random_state = 42,
            use_gpu=True # CPU mode is incredibly efficient for CSR matrices
        )

        print(" -> Commencing Matrix Factorization Training...")
        model.fit(sparse_matrix)

        # # ==========================================
        # # PHASE 5: EXPORT & CLEANUP
        # # ==========================================
        # print("\n[PHASE 5] Exporting Latent Vectors...")
        # os.makedirs(os.path.dirname(os.path.abspath(output_user_matrix)), exist_ok=True)
        # np.save(output_user_matrix, model.user_factors)

        # os.makedirs(os.path.dirname(os.path.abspath(output_item_matrix)), exist_ok=True)
        # np.save(output_item_matrix, model.item_factors)

        # # Dọn dẹp rác (Tùy chọn: Xóa thư mục temp_dir nếu muốn)
        # print(f"\n[Success] Big Data ALS complete! Matrices saved.")
        # print("You can safely delete the 'output/temp_bigdata/' folder to save disk space.")
        # 5. Extract and Export Latent Vectors
        print("Training complete. Extracting latent factor matrices...")

        # Kiểm tra và kéo dữ liệu từ GPU về Numpy Array trên CPU
        if hasattr(model.user_factors, 'to_numpy'):
            user_vectors = model.user_factors.to_numpy()
            item_vectors = model.item_factors.to_numpy()
            print(" -> Successfully pulled matrices from GPU to CPU.")
        else:
            user_vectors = model.user_factors
            item_vectors = model.item_factors

        print(f"Saving User Matrix -> {output_user_matrix}")
        os.makedirs(os.path.dirname(os.path.abspath(output_user_matrix)), exist_ok=True)
        np.save(output_user_matrix, user_vectors)

        print(f"Saving Item Matrix -> {output_item_matrix}")
        os.makedirs(os.path.dirname(os.path.abspath(output_item_matrix)), exist_ok=True)
        np.save(output_item_matrix, item_vectors)
# --- Cách gọi hàm ---
# BigDataCFBuilder.build_als_model_out_of_core(
#     interactions_path="data/train_interactions.csv",
#     user_index_path="output/user_index.csv",
#     item_index_path="output/item_index.csv",
#     output_user_matrix="output/cf_user_vectors.npy",
#     output_item_matrix="output/cf_item_vectors.npy"
# )

In [ ]:
import gc
i = 32
while i <= 128:
  CFBuilder.build_als_model(
      interactions_path=cfg.path_interactions,
      user_index_path=cfg.path_user_mapping,
      latent_factors = i,
      item_index_path= cfg.path_item_mapping,
      output_user_matrix= f'{cfg.out_dir}/cf_als_user_profiles_{i}.npy',
      output_item_matrix= f'{cfg.out_dir}/cf_als_item_matrix_{i}.npy'
      )
  gc.collect()
  i = i*2

INITIATING BIG DATA ALS PIPELINE (100M+ ROWS)

[PHASE 1] Executing DuckDB Out-of-Core JOIN & Math Calculation...
 -> Inspecting schema for item mapping...
 -> Detected item index column: 'row_idx'


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 -> DuckDB mapping complete! Saved clean dataset to: output/temp_bigdata/mapped_interactions.parquet

[PHASE 2] Streaming Parquet to Disk-Backed Memmap Arrays...
 -> Total valid interactions: 178,572,441 rows

 -> Memmap flush complete.

[PHASE 3] Building CSR Sparse Matrix...
 -> Matrix constraints: 743,401 Users x 1,173,713 Items
 -> CSR Matrix built successfully in RAM.

[PHASE 4] Initializing ALS Engine (Factors: 32)...
 -> Commencing Matrix Factorization Training...


  0%|          | 0/20 [00:00<?, ?it/s]

2026-04-23 19:14:34,316 - Final training loss 0.0026


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 -> DuckDB mapping complete! Saved clean dataset to: output/temp_bigdata/mapped_interactions.parquet

[PHASE 2] Streaming Parquet to Disk-Backed Memmap Arrays...
 -> Total valid interactions: 178,572,441 rows

 -> Memmap flush complete.

[PHASE 3] Building CSR Sparse Matrix...
 -> Matrix constraints: 743,401 Users x 1,173,713 Items
 -> CSR Matrix built successfully in RAM.

[PHASE 4] Initializing ALS Engine (Factors: 128)...
 -> Commencing Matrix Factorization Training...


  0%|          | 0/20 [00:00<?, ?it/s]

2026-04-23 19:26:14,203 - Final training loss 0.0021


Training complete. Extracting latent factor matrices...
 -> Successfully pulled matrices from GPU to CPU.
Saving User Matrix -> /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v3/cf_als_user_profiles_128.npy
Saving Item Matrix -> /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v3/cf_als_item_matrix_128.npy


In [ ]:
%pip install faiss-cpu
from scripts.faiss_builder import FaissBuilder

print(cfg.model_dir)
i = 32
while i <= 128:
  FaissBuilder.build_ivf_inner_product_index(matrix_path=f'{cfg.out_dir}/cf_als_item_matrix_{i}.npy',output_index_path=f'{cfg.out_dir}/cf_als_item_matrix__{i}.index')
  gc.collect()
  i = i*2

/content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1
Loading matrix from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v3/cf_als_item_matrix_32.npy...
Matrix loaded. Shape: (1,173,713, 32). Calculated nlist: 4333
Initializing IndexFlatIP as the base quantizer...
Initializing IndexIVFFlat structure...
Training the K-Means clustering (This may take a moment)...
Adding vectors to the trained index...
Saving the finalized index to /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v3/cf_als_item_matrix__32.index...
[Success] IVF Index successfully built and saved.
Loading matrix from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v3/cf_als_item_matrix_64.npy...
Matrix loaded. Shape: (1,173,713, 64). Calculated nlist: 4333
Initializing IndexFlatIP as the base quantizer...
Initializing IndexIVFFlat structure...
Training the K-Means clustering (This may take a moment)...
Adding vectors to the trained index...
Saving the fin

In [ ]:
%pip install -q faiss-gpu-cu12
%pip install -q duckdb polars

sys.path.append(os.path.abspath('/content/drive/My Drive/Thesis/book_recsys'))
sys.path.insert(0,'/content/drive/My Drive/Thesis/book_recsys/app')

from app.recommenders.vector_recommender import VectorRecommender
from app.recommenders.processer import BookProcessor, UserProcessor

_base_dir = '/content/drive/My Drive/Thesis/book_recsys'
_data_dir = f'{_base_dir}/data/processed_2'
_models_dir = f'{_data_dir}/model_v1'
_out_dir = f'{_data_dir}/model_v3'

user_index = f'{_models_dir}/user_index.csv'
imtem_index = f'{_models_dir}/cb_sbert_book_index.csv'
history_db = f'{_models_dir}/user_history.db'

sbert_item_matrix = f'{_models_dir}/cb_sbert_matrix.npy'
sbert_user_matrix = f'{_models_dir}/cb_sbert_user_matrix.npy'
sbert_item_faiss_index = f'{_models_dir}/cb_sbert_matrix.index'

ials_user_matrix = f'{_models_dir}/cf_als_user_profiles.npy'
ials_item_matrix = f'{_models_dir}/cf_als_item_matrix.npy'
ials_item_faiss_index = f'{_models_dir}/cf_als_item_matrix.index'

test_file = f'{_data_dir}/test_pos.csv'

import gc
gc.collect()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
import sys
import os
%pip install -q faiss-gpu-cu12
%pip install -q duckdb polars

sys.path.append(os.path.abspath('/content/drive/My Drive/Thesis/book_recsys'))
sys.path.insert(0,'/content/drive/My Drive/Thesis/book_recsys/app')

from app.recommenders.vector_recommender import VectorRecommender
from app.recommenders.processer import BookProcessor, UserProcessor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 9.0 MB/s eta 0:00:00


In [ ]:
_base_dir = '/content/drive/My Drive/Thesis/book_recsys'
_data_dir = f'{_base_dir}/data/processed_2'
_models_dir = f'{_data_dir}/model_v1'
_eval_dir = f'{_data_dir}/eval_v1'

user_index = f'{_models_dir}/user_index.csv'
imtem_index = f'{_models_dir}/cb_sbert_book_index.csv'
history_db = f'{_models_dir}/user_history.db'

sbert_item_matrix = f'{_models_dir}/cb_sbert_matrix.npy'
sbert_user_matrix = f'{_models_dir}/cb_sbert_user_matrix.npy'
sbert_item_faiss_index = f'{_models_dir}/cb_sbert_matrix.index'

ials_user_matrix = f'{_models_dir}/cf_als_user_profiles.npy'
ials_item_matrix = f'{_models_dir}/cf_als_item_matrix.npy'
ials_item_faiss_index = f'{_models_dir}/cf_als_item_matrix.index'

test_file = f'{_data_dir}/test_main.csv'
ground_truth_file = f'{_eval_dir}/test_ground_truth.json'

cbf_predictions = f'{_eval_dir}/cbf_prediction.json'
cf_predictions = f'{_eval_dir}/cf_prediction.json'

In [ ]:
import json
import time
import gc
# from processer import UserProcessor, BookProcessor
# from recommender import UniversalVectorRecommender

def get_chunks(lst, chunk_size=10_000):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

def append_predictions(predictions_dict, output_path, is_first_write=False):

    if not predictions_dict:
        return

    json_ready_data = {
        str(uid): [item.to_api_dict() for item in rec_items]
        for uid, rec_items in predictions_dict.items()
    }

    # Format từng key-value thành chuỗi chuẩn JSON
    items_str = []
    for k, v in json_ready_data.items():
        items_str.append(f'  "{k}": {json.dumps(v, ensure_ascii=False)}')

    with open(output_path, 'a', encoding='utf-8') as f:
        if not is_first_write:
            f.write(",\n")
        f.write(",\n".join(items_str))

def main():
    print("Start prediction")
    try:
      del cf_engine
      del cbf_engine
      del user_proc
      del book_proc
    except:
      pass
    gc.collect()
    # ==========================================
    # 1. Load user List
    # ==========================================
    gt_path = ground_truth
    with open(gt_path, 'r', encoding='utf-8') as f:
        ground_truth_data = json.load(f)
    test_users = list(ground_truth_data.keys())
    print(f"loading {len(test_users):,} users from Test.")

    # ==========================================
    # 2. Initializting
    # ==========================================
    print("-------Data Processors...")
    book_proc = BookProcessor(item_index_path= imtem_index)
    user_proc = UserProcessor(user_index_path= user_index, history_db_path= history_db)
    gc.collect()
    chunks = list(get_chunks(test_users, 5000))

    # ==========================================
    # 3.  CF (ALS)
    # ==========================================
    print("\n" + "="*40 + "\nRunning CF (ALS)...")
    cf_engine = VectorRecommender(
        book_processor= book_proc,
        user_processor= user_proc,
        faiss_index_path=ials_item_faiss_index,
        user_vectors_path = ials_user_matrix,
    )


    start_time = time.time()
    cf_output_path = f"{_eval_dir}/cf_test_predictions.json"

    # Khởi tạo file JSON với dấu mở ngoặc
    with open(cf_output_path, 'w', encoding='utf-8') as f:
        f.write("{\n")

    is_first_write = True
    for i, chunk in enumerate(chunks, 1):
        print(f" -> Processing {i}/{len(chunks)}...", end="\r")
        preds = cf_engine.recommend_batch(chunk, k=50)

        if preds:
            append_predictions(preds, cf_output_path, is_first_write)
            is_first_write = False

    # Đóng file JSON
    with open(cf_output_path, 'a', encoding='utf-8') as f:
        f.write("\n}\n")

    print(f"\nCF Done {time.time() - start_time:.2f} second.")
    del cf_engine
    gc.collect()

    i = 32
    while i <=128:
      print(f" -> Processing {i}/{128}...", end="\r")
      cf_engine = VectorRecommender(
        book_processor= book_proc,
        user_processor= user_proc,
        faiss_index_path=f'{_data_dir}/model_v3/cf_als_item_matrix__{i}.index',
        user_vectors_path = f'{_data_dir}/model_v3/cf_als_user_profiles_{i}.npy',
      )

      start_time = time.time()
      cf_output_path = f"{_eval_dir}/cf_test_predictions_{i}.json"

      # Khởi tạo file JSON với dấu mở ngoặc
      with open(cf_output_path, 'w', encoding='utf-8') as f:
          f.write("{\n")

      is_first_write = True
      for i, chunk in enumerate(chunks, 1):
          print(f" -> Processing {i}/{len(chunks)}...", end="\r")
          preds = cf_engine.recommend_batch(chunk, k=50)

          if preds:
              append_predictions(preds, cf_output_path, is_first_write)
              is_first_write = False

      # Đóng file JSON
      with open(cf_output_path, 'a', encoding='utf-8') as f:
          f.write("\n}\n")

      print(f"\nCF Done {time.time() - start_time:.2f} second.")
      del cf_engine
      gc.collect()
      i=i*2

    # ==========================================
    # 4. CBF (NLP)
    # ==========================================
    print("="*40 + "\nRunning CBF (NLP)...")
    cbf_engine = VectorRecommender(
        book_processor= book_proc,
        user_processor= user_proc,
        faiss_index_path=sbert_item_faiss_index,
        user_vectors_path = sbert_user_matrix,
    )

    start_time = time.time()
    cbf_output_path = f"{_eval_dir}/cbf_test_predictions.json"

    # Khởi tạo file JSON với dấu mở ngoặc
    with open(cbf_output_path, 'w', encoding='utf-8') as f:
        f.write("{\n")

    is_first_write = True
    for i, chunk in enumerate(chunks, 1):
        print(f" ---- Processing {i}/{len(chunks)}...", end="\r")
        preds = cbf_engine.recommend_batch(chunk, k=50)

        if preds:
            append_predictions(preds, cbf_output_path, is_first_write)
            is_first_write = False

    # Đóng file JSON
    with open(cbf_output_path, 'a', encoding='utf-8') as f:
        f.write("\n}\n")

    print(f"\nDone {time.time() - start_time:.2f} giây.")
    print("ALL DONE")

if __name__ == "__main__":
    main()


Start prediction
loading 743,401 users from Test.
-------Data Processors...
Loading Book mapping from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/cb_sbert_book_index.csv ...
 -> Loaded 1,173,713 books into RAM dictionary.
Loading User mapping from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/user_index.csv...
Connecting to Persistent User History DB...
Loading Data & CPU FAISS Index...
Cloning FAISS Index to GPU VRAM (for Batch Processing)...
Recommender Engine Ready (Users: 743,401)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


CF Done 3971.12 second.
Loading Data & CPU FAISS Index...
Cloning FAISS Index to GPU VRAM (for Batch Processing)...
Recommender Engine Ready (Users: 743,401)
